# 01 · Data download and quality control

**Goal:** get TCGA-LIHC gene counts and clinical data, decide which samples to keep, and check that the data looks right *before* any statistics.

Checklist for this notebook:
1. Download and load the data
2. Label samples as tumor / normal from the TCGA barcode
3. Check library sizes (sequencing depth)
4. Filter out genes with almost no reads
5. PCA: do tumor and normal separate? Any outliers?
6. Sanity check with known liver cancer marker genes
7. Save clean data for notebook 02

> 📝 Write your observations in the **Notes** cells. They become your README text later.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import data

FIG = ROOT / "figures"
FIG.mkdir(exist_ok=True)
data.PROCESSED.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
PALETTE = {"Tumor": "#c0392b", "Normal": "#2e86c1"}

## 1. Download and load

In [ ]:
data.download_all()   # skips files that are already there

In [ ]:
counts = data.load_counts()
gene_map = data.load_gene_map()
print(f"Counts matrix: {counts.shape[0]:,} genes x {counts.shape[1]} samples")
counts.iloc[:5, :4]

## 2. Label samples

TCGA barcodes encode the sample type: `TCGA-2V-A95S-`**`01`**`A`
- `01` = primary tumor
- `11` = solid tissue normal (adjacent non-tumor liver)
- `02` = recurrent tumor (we drop these)

Some samples were sequenced more than once (different vials/aliquots). We keep one per patient + type.

In [ ]:
meta = data.sample_table(counts.columns)
print(meta["condition"].value_counts(), "\n")

# keep only primary tumor and normal
meta = meta[meta["condition"].isin(["Tumor", "Normal"])]

# one sample per patient + condition (first vial alphabetically, e.g. A before B)
meta = meta.sort_values("vial")
meta = meta[~meta.duplicated(["patient", "condition"], keep="first")].sort_index()

per_patient = meta.groupby("patient")["condition"].nunique()
paired_patients = per_patient[per_patient == 2].index
meta["paired"] = meta["patient"].isin(paired_patients)

counts = counts[meta.index]
print(meta["condition"].value_counts())
print(f"Patients with both tumor and normal: {len(paired_patients)}")

## 3. Library sizes

Library size = total reads per sample. Very low or very high values can mean a failed or unusual sample.

In [ ]:
meta["lib_size_M"] = counts.sum(axis=0) / 1e6

fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(data=meta, x="lib_size_M", hue="condition", palette=PALETTE, bins=40, ax=ax)
ax.set(xlabel="Total reads (millions)", ylabel="Samples", title="Library size per sample")
plt.tight_layout(); plt.savefig(FIG / "01_library_sizes.png", dpi=150); plt.show()

meta.groupby("condition")["lib_size_M"].describe().round(1)

## 4. Filter low-count genes

~60k genes are annotated, but many have almost no reads (non-expressed genes, pseudogenes). They add noise and multiple-testing burden.

Rule: keep a gene if it has **≥ 10 counts in at least N samples**, where N = size of the smaller group (normals). This way a gene expressed only in normal tissue is still kept.

In [ ]:
min_samples = meta["condition"].value_counts().min()
keep = (counts >= 10).sum(axis=1) >= min_samples
counts_f = counts[keep]
print(f"Genes kept: {keep.sum():,} / {len(keep):,} ({keep.mean():.0%})")

## 5. PCA

For visualization only (not for DE testing), we normalize with log2 counts-per-million and use the 2,000 most variable genes.

**What to look for:** tumor and normal should mostly separate. Samples far from their group may be mislabeled, contaminated, or low quality.

In [ ]:
cpm = counts_f / counts_f.sum(axis=0) * 1e6
logcpm = np.log2(cpm + 1)
top_var = logcpm.var(axis=1).sort_values(ascending=False).index[:2000]
X = logcpm.loc[top_var].T
X = X - X.mean()

pca = PCA(n_components=10).fit(X)
pcs = pd.DataFrame(pca.transform(X)[:, :2], index=X.index, columns=["PC1", "PC2"]).join(meta)
ev = pca.explained_variance_ratio_ * 100

# flag outliers: > 3 SD from their group's centre on PC1/PC2
def distance_z(g):
    d = np.sqrt((g["PC1"] - g["PC1"].median())**2 + (g["PC2"] - g["PC2"].median())**2)
    return (d - d.mean()) / d.std()
pcs["dist_z"] = pcs.groupby("condition", group_keys=False)[["PC1", "PC2"]].apply(distance_z)
pcs["outlier"] = pcs["dist_z"] > 3

fig, ax = plt.subplots(figsize=(7, 5.5))
sns.scatterplot(data=pcs, x="PC1", y="PC2", hue="condition", palette=PALETTE, s=30, alpha=.75, ax=ax)
out = pcs[pcs["outlier"]]
ax.scatter(out["PC1"], out["PC2"], s=120, facecolors="none", edgecolors="black", label="flagged outlier")
for s, r in out.iterrows():
    ax.annotate(s[:16], (r["PC1"], r["PC2"]), fontsize=7, xytext=(4, 4), textcoords="offset points")
ax.set(xlabel=f"PC1 ({ev[0]:.1f}%)", ylabel=f"PC2 ({ev[1]:.1f}%)", title="PCA of TCGA-LIHC samples")
ax.legend(frameon=False)
plt.tight_layout(); plt.savefig(FIG / "01_pca.png", dpi=150); plt.show()

print("Flagged outliers:")
pcs.loc[pcs["outlier"], ["condition", "lib_size_M", "dist_z"]].round(2)

**Decide about outliers — don't auto-delete them.** For each flagged sample, check: is its library size odd? Does a *normal* sample sit inside the tumor cluster (could be tumor-contaminated)? Write your decision and reason below, then list the samples to drop in the next cell.

In [ ]:
DROP = []   # e.g. ["TCGA-XX-XXXX-11A"] — add with a reason in the Notes cell
meta = meta.drop(index=DROP)
counts_f = counts_f[meta.index]
print(meta["condition"].value_counts())

## 6. Sanity check with known markers

If labels and data are correct, well-known genes should behave as expected in HCC:
- **GPC3**, **AFP**: typically higher in tumor (AFP only in a subset of patients)
- **CYP2E1**, **CYP1A2**: normal liver metabolism enzymes, typically lower in tumor

If these go the *wrong* way, stop and check the sample labels.

In [ ]:
markers = ["GPC3", "AFP", "CYP2E1", "CYP1A2"]
symbol = gene_map.reindex(logcpm.index)
rows = [symbol[symbol == g].index[0] for g in markers if (symbol == g).any()]

long = (logcpm.loc[rows, meta.index].T
        .rename(columns=symbol.to_dict())
        .join(meta["condition"])
        .melt(id_vars="condition", var_name="gene", value_name="log2 CPM"))

g = sns.catplot(data=long, x="condition", y="log2 CPM", col="gene", kind="box",
                palette=PALETTE, hue="condition", col_wrap=4, height=3.2, sharey=False, legend=False)
g.figure.suptitle("Known HCC marker genes", y=1.04)
plt.savefig(FIG / "01_marker_check.png", dpi=150, bbox_inches="tight"); plt.show()

## 7. Clinical and survival data (quick look)

We'll use this properly in notebook 04. For now: how many patients, how many deaths (events), and how long the follow-up is.

In [ ]:
surv = data.load_survival()
print(surv.columns.tolist())
surv_t = surv[surv["sample"].isin(meta.index[meta["condition"] == "Tumor"])]
print(f"Tumor samples with survival data: {len(surv_t)}")
print(f"Deaths (OS = 1): {int(surv_t['OS'].sum())}")
print(f"Median follow-up: {surv_t['OS.time'].median() / 30.4:.1f} months")

In [ ]:
clin = data.load_clinical()
print(f"Clinical table: {clin.shape}")
# column names differ between Xena releases, so search for the ones we need
[c for c in clin.columns if any(k in c.lower() for k in ["stage", "grade", "gender", "age_at", "vital"])]

## 8. Save for notebook 02

In [ ]:
counts_f.to_csv(data.PROCESSED / "counts_filtered.csv.gz")
meta.to_csv(data.PROCESSED / "samples.csv")
gene_map.to_csv(data.PROCESSED / "gene_map.csv")
print("Saved:", *sorted(p.name for p in data.PROCESSED.glob("*.*")), sep="\n  ")

## Notes

_Fill these in after running — they go into the README._

- Final sample numbers: __ tumor, __ normal, __ paired patients
- Library sizes: ...
- PCA: do tumor and normal separate? What does PC1 seem to capture?
- Outliers flagged and what I decided: ...
- Marker genes behaved as expected? ...
- Anything surprising: ...